In [ ]:
# @title
# Exemplo simples sobre consistência forte e consistência eventual
import time
import threading
import random
from IPython.display import clear_output

# Simulação de um "banco de dados" replicado
replicas = {
    "Replica_A": 0,
    "Replica_B": 0,
    "Replica_C": 0
}

# Função para exibir o estado atual das réplicas
def show_replicas(title):
    clear_output(wait=True)
    print(f"\n=== {title} ===")
    for replica, valor in replicas.items():
        print(f"{replica}: {valor}")
    time.sleep(1)

CONSISTÊNCIA FORTE

In [ ]:
# @title
# CONSISTÊNCIA FORTE - Atualização de todas as réplicas ao mesmo tempo (síncrono)
def strong_consistency_update(value):
    for replica in replicas:
        replicas[replica] = value
    show_replicas("Consistência Forte (Atualização Síncrona)")

print("=== Demonstração: Consistência Forte ===")
for i in range(6):
    strong_consistency_update(i)
    time.sleep(1)

CONSISTÊNCIA EVENTUAL

In [ ]:
# @title
# CONSISTÊNCIA EVENTUAL - Atualização de réplicas em tempos diferentes (assíncrono)
def eventual_consistency_update(value):
    def delayed_update(replica, delay):
        time.sleep(delay)
        replicas[replica] = value

    threads = []
    for replica in replicas:
        delay = random.uniform(0.5, 2.5)  # delay gerado aleatóriamente
        t = threading.Thread(target=delayed_update, args=(replica, delay))
        t.start()
        threads.append(t)

    # Atualização das réplicas em tempos diferentes
    for _ in range(5):
        show_replicas("Consistência Eventual (Propagação Assíncrona)")
    for t in threads:
        t.join()  # espera até que todas as threads terminem as atualizações

print("\n\n=== Demonstração: Consistência Eventual ===")
for i in range(3, 6):
    eventual_consistency_update(i)
    time.sleep(1)


---

Questões sobre consistência:
1) Execute o código de consistência eventual e modifique o laço for referente a atualização das réplicas em tempos diferentes.

a) Mesmo que pareçam diferentes no início, o que garante que todas terminam com o mesmo valor?


---

2) No trecho do código:

```
delay = random.uniform(0.5, 2.5)
```

Altere os valores para simular situações diferentes:

- Rede lenta: delay = random.uniform(2, 5)

- Rede instável: delay = random.choice([0.25, 1, 4, 6])

- Falha em uma réplica: comente a atualização de uma delas (ex.: if replica != "Replica_C": ...)

Observe como isso afeta a convergência das réplicas e responda:

a) O que acontece se uma réplica nunca receber a atualização?

b) Em um sistema real, como o sistema poderia detectar e corrigir isso?

c) Qual seria o impacto para o usuário final se ele consultasse a réplica desatualizada?


In [ ]:
# @title
# CONSISTÊNCIA EVENTUAL - Atualização de réplicas em tempos diferentes (assíncrono)
def eventual_consistency_update(value):
    def delayed_update(replica, delay):
        time.sleep(delay)
        replicas[replica] = value

    threads = []
    for replica in replicas:
        if replica == "Replica_C":
            print(f"⚠️ {replica} está fora do ar! (falha simulada)")
            continue  # não cria thread para essa réplica

        delay = random.uniform(0.5, 2.5)  # delay gerado aleatóriamente
        t = threading.Thread(target=delayed_update, args=(replica, delay))
        t.start()
        threads.append(t)

    # Atualização das réplicas em tempos diferentes
    for _ in range(5):
        show_replicas("Consistência Eventual (Propagação Assíncrona)")
    for t in threads:
        t.join()  # espera até que todas as threads terminem as atualizações

print("\n\n=== Demonstração: Consistência Eventual ===")
for i in range(3, 6):
    eventual_consistency_update(i)
    time.sleep(1)


---

3) Execute a função de consistência forte do código e após, a consistência eventual.

Feito isso, adicione um cronômetro simples antes e após a execução de cada função:


```
start = time.time()
strong_consistency_update(10)
print("Tempo forte:", time.time() - start)
```
e

```
start = time.time()
eventual_consistency_update(10)
print("Tempo eventual:", time.time() - start)
```

Dado isso, responda:

a) Em um sistema real, qual modelo tende a oferecer uma resposta mais rápida ao cliente?

b) E qual o comportamente apresentado? É compatível com a realidade?

In [ ]:
# @title
start = time.time()
strong_consistency_update(10)
print("Tempo forte:", time.time() - start)

In [ ]:
# @title
start = time.time()
strong_consistency_update(10)
print("Tempo forte:", time.time() - start)


---
4) Simule uma falha “temporária” (réplica com atualização lenta em comparação as demais). Em vez de pular a atualização, no código faça uma auteração que aumente o atraso de uma réplica específica.

In [ ]:
# @title
start = time.time()
strong_consistency_update(10)
print("Tempo forte:", time.time() - start)

In [ ]:
def eventual_consistency_update(value):
    def delayed_update(replica, delay):
        time.sleep(delay)
        replicas[replica] = value

    threads = []
    for replica in replicas:
      if replica == "Replica_C":
          delay = random.uniform(5, 8)  # atraso maior que as demais
          print(f"{replica} está apresentando lentidão... (atraso = {delay:.1f}s)")
      else:
          delay = random.uniform(0.5, 2.0)

      t = threading.Thread(target=delayed_update, args=(replica, delay))
      t.start()
      threads.append(t)

    # Atualização das réplicas em tempos diferentes
    for _ in range(5):
        show_replicas("Consistência Eventual (Propagação Assíncrona)")
    for t in threads:
        t.join()  # espera até que todas as threads terminem as atualizações

print("\n\n=== Demonstração: Consistência Eventual ===")
for i in range(0, 6):
    eventual_consistency_update(i)
    time.sleep(1)



=== Consistência Eventual (Propagação Assíncrona) ===
Replica_A: 5
Replica_B: 5
Replica_C: 4



---

Consistência Sequencial

In [ ]:
# @title
import time
from IPython.display import clear_output

data_store = {"x": "NIL"}  # valor inicial de x (dado)
history = []               # histórico global das operações

# -----------------------------
# Funções auxiliares
# -----------------------------
def show_state(title):
    #clear_output(wait=True)
    print(f"\n=== {title} ===\n")
    print(f"Valor atual de x: {data_store['x']}\n")
    print("Histórico global das operações:")
    for operacao in history:
        print(" ->", operacao)
    time.sleep(1)

# função que simula uma operação de escrita
def write(process, value):
    operacao = f"W{process}(x){value}"
    data_store["x"] = value
    history.append(operacao)
    show_state(f"{operacao} — Escrita por P{process}")

# função que simula uma operação de leitura
def read(process):
    value = data_store["x"]
    operacao = f"R{process}(x){value}"
    history.append(operacao)
    show_state(f"{operacao} — Leitura por P{process}")
    #return value

In [ ]:
# @title
# Exemplo sem concorrência
print("=== Exemplo 1: Sem concorrência entre processos ===")
time.sleep(1)
write(1, "a")   # P1 escreve 'a' em x
read(2)         # P2 lê x como 'a'

In [ ]:
# @title
# Exemplo com concorrência
print("\n\n=== Exemplo 2: Com concorrência entre processos (intercalando operações) ===")
print("\n\n=============== Com escrita local e propagação atrasada ====================")
time.sleep(1)
history.clear()
data_store["x"] = "NIL"

# P1 escreve 'a' com atualização local
data_store["x"] = "a"
history.append("W1(x)a [local]")
show_state("Atualização local de P1")
time.sleep(5)

# P2 lê antes da propagação
data_store["x"] = "NIL"
read(2)
time.sleep(5)

# Atualização de P1 é propagada
data_store["x"] = "a"
read(2)
time.sleep(5)

Consistência Sequencial com Threads

In [ ]:
# @title
import threading
import time
import random
from IPython.display import clear_output

global_store = {"x": "NIL"}  # valor global compartilhado
replicas = {"P1": {"x": "NIL"},"P2": {"x": "NIL"}}
history = []

def show_state(title):
    clear_output(wait=True)
    print(f"\n=== {title} ===\n")
    print(f"Global: x = {global_store['x']}")
    for p in replicas:
        print(f"{p}: x = {replicas[p]['x']}")
    print("\nHistórico global:")
    for op in history:
        print(" →", op)
    time.sleep(5)

def write(process, value):
    replicas[process]["x"] = value
    operacao = f"W{process[-1]}(x){value} [local]"
    history.append(operacao)
    show_state(f"{operacao} — Escrita local")

    delay = random.uniform(1, 3)
    def propagate():
        time.sleep(delay)
        global_store["x"] = value
        for p in replicas:
            replicas[p]["x"] = value
        history.append(f"Propagação de {process}: x={value} (atraso {delay:.1f}s)")
        show_state(f"Propagação concluída de {process}")
    threading.Thread(target=propagate).start()

def read(process):
    value = replicas[process]["x"]
    operacao = f"R{process[-1]}(x){value}"
    history.append(operacao)
    show_state(f"{operacao} — Leitura por {process}")
    return value

def reset():  # Reinicia o global_store e o history
    global_store["x"] = "NIL"
    for p in replicas:
        replicas[p]["x"] = "NIL"
    history.clear()

def process_1():
    write("P1", "a")
    time.sleep(random.uniform(1, 2))
    read("P1")

def process_2():
    time.sleep(random.uniform(0.5, 1.5))
    read("P2")
    time.sleep(random.uniform(2, 3))
    write("P2", "b")
    read("P2")

In [ ]:
# @title
print("=== Simulando consistência sequencial com atrasos ===")
time.sleep(2)
reset()

t1 = threading.Thread(target=process_1)
t2 = threading.Thread(target=process_2)

t1.start()
t2.start()
t1.join()
t2.join()

time.sleep(2)
print("\n✅ Execução concluída.")